# Week 4: High-Learning Joint Always-On LoRA Training (V2)

This revised notebook trains one always-enabled LoRA adapter on:

- 700 total synthetic training examples covering 500 unique facts
- 140 general training questions, replayed to protect and improve general capability
- Qwen2.5-1.5B-Instruct in 4-bit, which remains compatible with a T4 GPU
- the rank-16, all-projection LoRA recipe that previously produced high
  held-out synthetic-fact accuracy

The notebook evaluates all 200 general questions before and after training and saves
separate train, validation, and test CSVs. The 40-question test split is the clean
held-out generalization result; the all-200 score is descriptive because 140 questions
are used for training. Final synthetic evaluation uses 800 held-out paraphrases.

## 0. Colab GPU Setup

In [ ]:
%pip uninstall -q -y bitsandbytes
%pip install -q -U "transformers==4.48.3" "accelerate==1.3.0" "peft==0.14.0" "datasets==3.2.0" sentencepiece "pandas==2.2.3"
%pip install -q --no-deps "bitsandbytes==0.49.2"

## 1. Mount Drive and Set Paths

In [ ]:
# GitHub-backed Colab setup. Keep GITHUB_TOKEN in Colab Secrets.
from pathlib import Path
import sys
import urllib.request

HELPER_PATH = Path('/content/github_colab_sync.py')
HELPER_URL = 'https://raw.githubusercontent.com/HannanSeyfi/unlearning-thesis/main/Tools/github_colab_sync.py'
if not HELPER_PATH.exists():
    urllib.request.urlretrieve(HELPER_URL, HELPER_PATH)
if str(HELPER_PATH.parent) not in sys.path:
    sys.path.insert(0, str(HELPER_PATH.parent))

from github_colab_sync import commit_and_push, resolve_week35_baseline_dir, setup_colab_repo

THESIS_DIR = setup_colab_repo()

GENERAL_DIR = (
    THESIS_DIR / 'Week 4 - Joint Training Experiments' / 'data' / 'general_knowledge_200_v1'
)
SYNTHETIC_DIR = (
    THESIS_DIR / 'Week 4 - Joint Training Experiments' / 'data' / 'synthetic_examples_700_v2'
)
RUN_DIR = (
    THESIS_DIR / 'Week 4 - Joint Training Experiments' / 'results'
    / 'joint_lora_700synthetic_general_v2'
)
ADAPTER_DIR = RUN_DIR / 'adapter'
CHECKPOINT_DIR = RUN_DIR / 'checkpoint_selection'
OUTPUT_DIR = RUN_DIR / 'results'
for folder in [ADAPTER_DIR, CHECKPOINT_DIR, OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

required = [
    GENERAL_DIR / 'manifest.json',
    GENERAL_DIR / 'general_all.jsonl',
    GENERAL_DIR / 'general_train.jsonl',
    GENERAL_DIR / 'general_validation.jsonl',
    GENERAL_DIR / 'general_test.jsonl',
    SYNTHETIC_DIR / 'manifest.json',
    SYNTHETIC_DIR / 'train_all.jsonl',
    SYNTHETIC_DIR / 'eval_forget.jsonl',
    SYNTHETIC_DIR / 'eval_retain.jsonl',
]
for path in required:
    assert path.exists(), (
        f'Missing data file: {path}. Run week4_create_joint_training_data.ipynb first.'
    )
print('Results:', RUN_DIR)


## 2. Imports and Configuration

In [ ]:
import gc
import hashlib
import json
import random
import re
import shutil
import unicodedata
from datetime import datetime, timezone

import bitsandbytes as bnb
import numpy as np
import pandas as pd
import torch
from bitsandbytes.cextension import lib as bnb_native_lib
from datasets import Dataset
from peft import (
    LoraConfig, PeftModel, TaskType, get_peft_model,
    prepare_model_for_kbit_training,
)
from peft.tuners.lora import LoraLayer
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
    DataCollatorForSeq2Seq, Trainer, TrainerCallback, TrainingArguments,
)

SEED = 42
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
TARGET_PERCENTAGE = 85.0
MAX_LENGTH = 192
MAX_EPOCHS = 20
LEARNING_RATE = 2e-4
GENERAL_REPETITIONS = 4
PER_DEVICE_BATCH_SIZE = 2
GRADIENT_ACCUMULATION_STEPS = 8
SELECTION_EPOCHS = [8, 12, 16, 20]
ADAPTER_STRENGTHS = [0.70, 0.85, 1.00]
MIN_SYNTHETIC_SELECTION_PERCENTAGE = 80.0
EARLY_STOP_SYNTHETIC_PERCENTAGE = 90.0
EARLY_STOP_GENERAL_PERCENTAGE = 85.0
LORA_R = 16
LORA_ALPHA = 32
LORA_TARGETS = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

SYNTHETIC_SYSTEM_PROMPT = (
    'You answer questions about fictional synthetic people using the provided learned facts.'
)
GENERAL_SYSTEM_PROMPT = (
    'Answer the question concisely. Return only the requested answer without explanation.'
)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
torch.cuda.manual_seed_all(SEED)
assert bnb_native_lib is not None, 'Restart runtime and rerun package setup.'
print('GPU:', torch.cuda.get_device_name(0))
print('bitsandbytes:', bnb.__version__)

## 3. Load and Verify Versioned Data

In [ ]:
def read_jsonl(path):
    return [
        json.loads(line)
        for line in Path(path).read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def verify_manifest(folder):
    manifest = json.loads((folder / 'manifest.json').read_text(encoding='utf-8'))
    for details in manifest['sets'].values():
        path = folder / details['jsonl_file']
        assert sha256_file(path) == details['jsonl_sha256'], (
            f'Checksum mismatch: {path}'
        )
    return manifest

general_manifest = verify_manifest(GENERAL_DIR)
synthetic_manifest = verify_manifest(SYNTHETIC_DIR)

general_all = read_jsonl(GENERAL_DIR / 'general_all.jsonl')
general_train = read_jsonl(GENERAL_DIR / 'general_train.jsonl')
general_validation = read_jsonl(GENERAL_DIR / 'general_validation.jsonl')
general_test = read_jsonl(GENERAL_DIR / 'general_test.jsonl')
synthetic_train = read_jsonl(SYNTHETIC_DIR / 'train_all.jsonl')
eval_forget = read_jsonl(SYNTHETIC_DIR / 'eval_forget.jsonl')
eval_retain = read_jsonl(SYNTHETIC_DIR / 'eval_retain.jsonl')

assert len(general_all) == 200
assert len(general_train) == 140
assert len(general_validation) == 20
assert len(general_test) == 40
assert len(synthetic_train) == 700
assert len(eval_forget) == 160
assert len(eval_retain) == 640

general_prompt_sets = [
    {row['prompt'].lower() for row in rows}
    for rows in [general_train, general_validation, general_test]
]
assert general_prompt_sets[0].isdisjoint(general_prompt_sets[1])
assert general_prompt_sets[0].isdisjoint(general_prompt_sets[2])
assert general_prompt_sets[1].isdisjoint(general_prompt_sets[2])
assert {row['prompt'].lower() for row in general_all} == set().union(
    *general_prompt_sets
), 'general_all.jsonl must contain the complete 200-question split.'

def synthetic_eval_record(row, split):
    return {
        'test_set': 'synthetic',
        'category': row['fact_type'],
        'eval_split': split,
        'example_id': row['example_id'],
        'prompt': row['prompt'],
        'expected_value': str(row['fact_value']),
    }

synthetic_final = (
    [synthetic_eval_record(row, 'forget') for row in eval_forget]
    + [synthetic_eval_record(row, 'retain') for row in eval_retain]
)
print('Loaded all datasets successfully.')

## 4. Evaluation Functions

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

NUMBER_WORDS = {
    'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
    'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
    'ten': '10', 'eleven': '11', 'twelve': '12', 'thirteen': '13',
    'fourteen': '14', 'fifteen': '15', 'sixteen': '16',
    'seventeen': '17', 'eighteen': '18', 'nineteen': '19',
    'twenty': '20',
}

def normalize_text(text):
    text = unicodedata.normalize('NFKD', str(text))
    text = ''.join(char for char in text if not unicodedata.combining(char))
    text = re.sub(r'\s+', ' ', text.strip().lower())
    for word, number in NUMBER_WORDS.items():
        text = re.sub(rf'(?<!\w){word}(?!\w)', number, text)
    return text.strip(' .,!?:;"\'')

def contains_value(generated, expected):
    generated = normalize_text(generated)
    expected = normalize_text(expected)
    return bool(
        expected
        and re.search(rf'(?<!\w){re.escape(expected)}(?!\w)', generated)
    )

def system_prompt(test_set):
    return (
        SYNTHETIC_SYSTEM_PROMPT
        if test_set == 'synthetic'
        else GENERAL_SYSTEM_PROMPT
    )

def load_base_model():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=quantization, device_map='auto'
    )
    model.eval()
    return model

@torch.inference_mode()
def generate_answer(model, prompt, test_set):
    text = tokenizer.apply_chat_template(
        [
            {'role': 'system', 'content': system_prompt(test_set)},
            {'role': 'user', 'content': prompt},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    output = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    tokens = output[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(tokens, skip_special_tokens=True).strip()

def evaluate(model, records, stage, progress_every=100):
    rows = []
    for index, row in enumerate(records, start=1):
        answer = generate_answer(model, row['prompt'], row['test_set'])
        expected = normalize_text(row['expected_value'])
        normalized = normalize_text(answer)
        rows.append({
            **row,
            'model_stage': stage,
            'generated_answer': answer,
            'exact_match': normalized == expected,
            'contains_value': contains_value(normalized, expected),
        })
        if progress_every and index % progress_every == 0:
            print(f'{stage}: {index}/{len(records)}')
    return pd.DataFrame(rows)

def set_lora_strength(model, strength):
    for module in model.modules():
        if isinstance(module, LoraLayer):
            for adapter_name in list(module.scaling):
                module.set_scale(adapter_name, strength)

## 5. Evaluate the Untouched Base Model

In [ ]:
base_model = load_base_model()
base_synthetic_df = evaluate(
    base_model, synthetic_final, 'base_before_training'
)
base_general_all_df = evaluate(
    base_model, general_all, 'base_before_training', progress_every=50
)
base_general_train_df = base_general_all_df[
    base_general_all_df['eval_split'] == 'train'
].copy()
base_general_validation_df = base_general_all_df[
    base_general_all_df['eval_split'] == 'validation'
].copy()
base_general_test_df = base_general_all_df[
    base_general_all_df['eval_split'] == 'test'
].copy()

base_synthetic_df.to_csv(OUTPUT_DIR / 'base_synthetic_results.csv', index=False)
base_general_all_df.to_csv(
    OUTPUT_DIR / 'base_general_all_200_results.csv', index=False
)
base_general_train_df.to_csv(
    OUTPUT_DIR / 'base_general_train_results.csv', index=False
)
base_general_test_df.to_csv(
    OUTPUT_DIR / 'base_general_test_results.csv', index=False
)
base_general_validation_df.to_csv(
    OUTPUT_DIR / 'base_general_validation_results.csv', index=False
)

BASE_VALIDATION_PERCENTAGE = (
    100 * base_general_validation_df['contains_value'].mean()
)
GENERAL_SELECTION_FLOOR = max(
    TARGET_PERCENTAGE, BASE_VALIDATION_PERCENTAGE - 5.0
)
print('Base general all 200:', f"{100 * base_general_all_df['contains_value'].mean():.2f}%")
print('Base held-out general test (40):', f"{100 * base_general_test_df['contains_value'].mean():.2f}%")
print('Base validation:', f'{BASE_VALIDATION_PERCENTAGE:.2f}%')
print('Selection floor:', f'{GENERAL_SELECTION_FLOOR:.2f}%')

del base_model
gc.collect()
torch.cuda.empty_cache()

## 6. Build the Joint Training Dataset

In [ ]:
def train_row(prompt, target, test_set):
    return {'prompt': prompt, 'target': str(target), 'test_set': test_set}

joint_rows = [
    train_row(row['prompt'], row['fact_value'], 'synthetic')
    for row in synthetic_train
]
joint_rows += [
    train_row(row['prompt'], row['expected_value'], 'general_knowledge')
    for _ in range(GENERAL_REPETITIONS)
    for row in general_train
]
random.Random(SEED).shuffle(joint_rows)

def tokenize_row(row):
    messages = [
        {'role': 'system', 'content': system_prompt(row['test_set'])},
        {'role': 'user', 'content': row['prompt']},
        {'role': 'assistant', 'content': row['target']},
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages[:-1], tokenize=False, add_generation_prompt=True
    )
    full_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    full = tokenizer(
        full_text, truncation=True, max_length=MAX_LENGTH, padding=False
    )
    prompt = tokenizer(
        prompt_text, truncation=True, max_length=MAX_LENGTH, padding=False
    )
    labels = full['input_ids'].copy()
    prompt_len = min(len(prompt['input_ids']), len(labels))
    labels[:prompt_len] = [-100] * prompt_len
    full['labels'] = labels
    return full

train_dataset = Dataset.from_list(joint_rows).map(
    tokenize_row, remove_columns=['prompt', 'target', 'test_set']
)

# Seen synthetic prompts are used only for checkpoint selection.
selection_df = pd.DataFrame(synthetic_train)
synthetic_selection = []
for split in ['forget', 'retain']:
    for fact_type in sorted(selection_df['fact_type'].unique()):
        subset = selection_df[
            (selection_df['split'] == split)
            & (selection_df['fact_type'] == fact_type)
        ].head(10)
        for _, row in subset.iterrows():
            synthetic_selection.append({
                'test_set': 'synthetic',
                'category': row['fact_type'],
                'eval_split': row['split'],
                'example_id': f"selection_{row['example_id']}",
                'prompt': row['prompt'],
                'expected_value': str(row['fact_value']),
            })

print('Synthetic training rows:', len(synthetic_train))
print('General training rows after repetition:', len(general_train) * GENERAL_REPETITIONS)
print('Joint rows:', len(joint_rows))

## 7. Train and Select the Best Always-On Checkpoint

In [ ]:
model = load_base_model()
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True
)
model.gradient_checkpointing_enable()
model = get_peft_model(
    model,
    LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        bias='none',
        target_modules=LORA_TARGETS,
    ),
)
model.print_trainable_parameters()

class JointSelectionCallback(TrainerCallback):
    def __init__(self):
        self.rows = []
        self.best_score = float('-inf')
        self.best = None

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        epoch = int(round(state.epoch))
        if epoch not in SELECTION_EPOCHS:
            return control
        was_training = model.training
        model.config.use_cache = True
        model.eval()
        for strength in ADAPTER_STRENGTHS:
            set_lora_strength(model, strength)
            synthetic_df = evaluate(
                model, synthetic_selection,
                f'epoch_{epoch}_strength_{strength}_synthetic',
                progress_every=0,
            )
            general_df = evaluate(
                model, general_validation,
                f'epoch_{epoch}_strength_{strength}_general',
                progress_every=0,
            )
            synthetic_pct = 100 * synthetic_df['contains_value'].mean()
            general_pct = 100 * general_df['contains_value'].mean()
            harmonic = (
                2 * synthetic_pct * general_pct / (synthetic_pct + general_pct)
                if synthetic_pct + general_pct else 0.0
            )
            eligible = (
                synthetic_pct >= MIN_SYNTHETIC_SELECTION_PERCENTAGE
            )
            targets_met = (
                synthetic_pct >= TARGET_PERCENTAGE
                and general_pct >= GENERAL_SELECTION_FLOOR
            )
            # An adapter that has not learned the synthetic facts cannot win.
            score = (
                1000 + harmonic if targets_met
                else harmonic if eligible
                else -1.0
            )
            row = {
                'epoch': epoch,
                'adapter_strength': strength,
                'synthetic_seen_percentage': synthetic_pct,
                'general_validation_percentage': general_pct,
                'harmonic_score': harmonic,
                'synthetic_learning_eligible': eligible,
                'selection_targets_met': targets_met,
                'selection_score': score,
            }
            self.rows.append(row)
            print(row)
            if eligible and score > self.best_score:
                self.best_score = score
                self.best = row.copy()
                set_lora_strength(model, 1.0)
                model.save_pretrained(ADAPTER_DIR)
                tokenizer.save_pretrained(ADAPTER_DIR)
        early_stop = any(
            row['epoch'] == epoch
            and row['synthetic_seen_percentage'] >= EARLY_STOP_SYNTHETIC_PERCENTAGE
            and row['general_validation_percentage'] >= EARLY_STOP_GENERAL_PERCENTAGE
            for row in self.rows
        )
        set_lora_strength(model, 1.0)
        model.config.use_cache = False
        if was_training:
            model.train()
        if early_stop:
            control.should_training_stop = True
            print('Early-stop balance targets reached.')
        return control

callback = JointSelectionCallback()
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir=str(CHECKPOINT_DIR / 'trainer_state'),
        seed=SEED,
        num_train_epochs=MAX_EPOCHS,
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        warmup_ratio=0.05,
        fp16=True,
        gradient_checkpointing=True,
        optim='paged_adamw_8bit',
        logging_steps=20,
        save_strategy='no',
        report_to='none',
        remove_unused_columns=False,
    ),
    train_dataset=train_dataset,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer=tokenizer, model=model, padding=True
    ),
    callbacks=[callback],
)
train_result = trainer.train()
selection_metrics_df = pd.DataFrame(callback.rows)
selection_metrics_df.to_csv(
    OUTPUT_DIR / 'checkpoint_selection_metrics.csv', index=False
)
if callback.best is None:
    diagnostic = selection_metrics_df.sort_values(
        ['synthetic_seen_percentage', 'general_validation_percentage'],
        ascending=False,
    ).iloc[0]
    raise RuntimeError(
        'No checkpoint learned the synthetic facts sufficiently. '
        f"Best seen-synthetic score: {diagnostic['synthetic_seen_percentage']:.2f}%. "
        'Stopping instead of evaluating an unlearned adapter.'
    )
print('Selected:', callback.best)

## 8. Final Held-Out Evaluation

In [ ]:
selected = callback.best
selected_strength = float(selected['adapter_strength'])
del trainer, model
gc.collect()
torch.cuda.empty_cache()

model = load_base_model()
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
set_lora_strength(model, selected_strength)
model.eval()

final_synthetic_df = evaluate(
    model, synthetic_final, 'joint_lora_after_training'
)
final_general_all_df = evaluate(
    model, general_all, 'joint_lora_after_training', progress_every=50
)
final_general_train_df = final_general_all_df[
    final_general_all_df['eval_split'] == 'train'
].copy()
final_general_validation_df = final_general_all_df[
    final_general_all_df['eval_split'] == 'validation'
].copy()
final_general_test_df = final_general_all_df[
    final_general_all_df['eval_split'] == 'test'
].copy()
final_synthetic_df.to_csv(
    OUTPUT_DIR / 'final_synthetic_results.csv', index=False
)
final_general_all_df.to_csv(
    OUTPUT_DIR / 'final_general_all_200_results.csv', index=False
)
final_general_train_df.to_csv(
    OUTPUT_DIR / 'final_general_train_results.csv', index=False
)
final_general_validation_df.to_csv(
    OUTPUT_DIR / 'final_general_validation_results.csv', index=False
)
final_general_test_df.to_csv(
    OUTPUT_DIR / 'final_general_test_results.csv', index=False
)

(ADAPTER_DIR / 'selected_config.json').write_text(
    json.dumps({
        'base_model_id': MODEL_ID,
        'selected_epoch': int(selected['epoch']),
        'selected_adapter_strength': selected_strength,
        'routing_used': False,
        'adapter_must_remain_enabled': True,
    }, indent=2),
    encoding='utf-8',
)

## 9. Save Results and Summaries

In [ ]:
all_results = pd.concat([
    base_synthetic_df,
    base_general_all_df,
    final_synthetic_df,
    final_general_all_df,
], ignore_index=True)

summary = (
    all_results
    .groupby(['model_stage', 'test_set', 'eval_split'])
    .agg(
        num_questions=('contains_value', 'size'),
        num_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda x: 100 * x.mean()),
        exact_match_percentage=('exact_match', lambda x: 100 * x.mean()),
    )
    .reset_index()
)
category_summary = (
    all_results
    .groupby(['model_stage', 'test_set', 'eval_split', 'category'])
    .agg(
        num_questions=('contains_value', 'size'),
        num_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda x: 100 * x.mean()),
        exact_match_percentage=('exact_match', lambda x: 100 * x.mean()),
    )
    .reset_index()
)

def percentage(frame, split=None):
    subset = frame if split is None else frame[frame['eval_split'] == split]
    return float(100 * subset['contains_value'].mean())

targets = pd.DataFrame([
    {'metric': 'synthetic_forget', 'percentage': percentage(final_synthetic_df, 'forget')},
    {'metric': 'synthetic_retain', 'percentage': percentage(final_synthetic_df, 'retain')},
    {'metric': 'general_test', 'percentage': percentage(final_general_test_df)},
])
targets['target_percentage'] = TARGET_PERCENTAGE
targets['target_met'] = targets['percentage'] >= TARGET_PERCENTAGE

all_results.to_csv(OUTPUT_DIR / 'all_test_results.csv', index=False)
summary.to_csv(OUTPUT_DIR / 'percentage_summary.csv', index=False)
category_summary.to_csv(
    OUTPUT_DIR / 'percentage_by_category.csv', index=False
)
targets.to_csv(OUTPUT_DIR / 'target_achievement.csv', index=False)

metrics = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'run_name': 'joint_lora_700synthetic_general_v2',
    'base_model_id': MODEL_ID,
    'routing_used': False,
    'training': {
        'synthetic_training_examples': len(synthetic_train), 'synthetic_unique_facts': len({(row['entity_id'], row['fact_type']) for row in synthetic_train}),
        'general_unique_train_questions': len(general_train),
        'general_repetitions': GENERAL_REPETITIONS,
        'joint_training_rows': len(joint_rows),
        'epochs': MAX_EPOCHS,
        'learning_rate': LEARNING_RATE,
        'lora_r': LORA_R,
        'lora_alpha': LORA_ALPHA,
        'lora_targets': LORA_TARGETS,
        'minimum_synthetic_selection_percentage': MIN_SYNTHETIC_SELECTION_PERCENTAGE,
        'early_stop_synthetic_percentage': EARLY_STOP_SYNTHETIC_PERCENTAGE,
        'early_stop_general_percentage': EARLY_STOP_GENERAL_PERCENTAGE,
    },
    'selection': selected,
    'base': {
        'synthetic_forget': percentage(base_synthetic_df, 'forget'),
        'synthetic_retain': percentage(base_synthetic_df, 'retain'),
        'general_all_200_descriptive': percentage(base_general_all_df),
        'general_train': percentage(base_general_train_df),
        'general_validation': percentage(base_general_validation_df),
        'general_test_held_out': percentage(base_general_test_df),
    },
    'after_training': {
        'synthetic_forget': percentage(final_synthetic_df, 'forget'),
        'synthetic_retain': percentage(final_synthetic_df, 'retain'),
        'general_all_200_descriptive': percentage(final_general_all_df),
        'general_train_seen': percentage(final_general_train_df),
        'general_validation': percentage(final_general_validation_df),
        'general_test_held_out': percentage(final_general_test_df),
    },
    'reporting_note': (
        'The all-200 score includes 140 training questions after fine-tuning. '
        'Use general_test_held_out as the thesis generalization result.'
    ),
    'all_85_targets_met': bool(targets['target_met'].all()),
}
(OUTPUT_DIR / 'metrics.json').write_text(
    json.dumps(metrics, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

display(targets)
display(summary)
display(category_summary)
print('All 85% targets met:', metrics['all_85_targets_met'])
print('Saved to:', RUN_DIR)

## Saved Output

All adapters, detailed predictions, checkpoint-selection metrics, percentages,
category summaries, and `metrics.json` are saved under:

`MyDrive/Thesis/Week 4 - Joint Training Experiments/results/joint_lora_700synthetic_general_v2`

In [ ]:
# GitHub output sync
commit_and_push(
    [RUN_DIR],
    'Colab: save Week 4 joint training outputs',
    repo_dir=THESIS_DIR,
)
